# DhakaFlow P1 — Perception Gate (Colab)

**Runs on Colab Pro GPU. Nothing here trains on your laptop.**

This notebook is *orchestration only* — all logic lives in the repo
(`research/perception/dataset_audit.py`, `dhakaflow/perception/taxonomy.py`),
so results are reproducible outside Colab and reviewable in git.

Order of work:
1. Environment + GPU report
2. Secure credentials (Colab Secrets — **never paste a token into a cell**)
3. Fetch datasets to Drive (BNVD 6.5 GB, Poribohon-BD license-clean)
4. **Audit real files** — measured counts, not paper claims
5. Clone official RT-DETR at the pinned SHA
6. COCO-pretrained baseline on Bangladeshi traffic (expected to miss rickshaw/CNG)

**Evidence rule:** every number saved here is measured. Where our measurement
disagrees with the paper, the measurement wins and the disagreement is recorded.


## 1 · Environment and GPU

In [ ]:
import subprocess, sys, platform, json, os
print("python :", platform.python_version())
try:
    import torch
    print("torch  :", torch.__version__)
    print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print("gpu    :", p.name)
        print("vram   : %.1f GB" % (p.total_memory / 1024**3))
except ImportError:
    print("torch not installed yet")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "no nvidia-smi")

## 2 · Mount Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/dhakaflow'
DATA_ROOT  = f'{DRIVE_ROOT}/data'
ART_ROOT   = f'{DRIVE_ROOT}/artifacts'
import os
for d in (DRIVE_ROOT, DATA_ROOT, ART_ROOT):
    os.makedirs(d, exist_ok=True)
print("drive ready:", DRIVE_ROOT)

In [ ]:
# Clone YOUR repo (replace with your remote), or upload it.
REPO_DIR = '/content/thesis'
import os, subprocess
if not os.path.exists(REPO_DIR):
    # subprocess.run(['git','clone','<YOUR_REPO_URL>', REPO_DIR], check=True)
    print('Set your repo URL above, or upload the repo folder to Colab.')
else:
    print('repo present')
import sys
sys.path.insert(0, REPO_DIR)

## 3 · Credentials — Colab Secrets only

Put your Kaggle token in the Colab **Secrets** panel (🔑 in the left sidebar) as
`KAGGLE_KEY` and `KAGGLE_USERNAME`. Never paste a token into a notebook cell —
notebooks get committed, shared and screenshotted.

**Rotate the token that was exposed earlier before using it here.**

In [ ]:
from google.colab import userdata
import os
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    print('kaggle credentials loaded from Colab Secrets')
except Exception as e:
    print('Set KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets first:', e)
!pip -q install kaggle pillow

## 4 · Fetch datasets to Drive

* **BNVD** — 6.5 GB, 17 classes. License field reads `unknown` on both Kaggle
  and the authors' README ⇒ used for research/benchmarking, **never
  redistributed**, and author permission is being sought in parallel.
* **Poribohon-BD** — CC BY 4.0, license-clean, keeps P1 unblocked regardless.

In [ ]:
BNVD_DIR = f'{DATA_ROOT}/BNVD'
import os, subprocess
if not os.path.exists(BNVD_DIR) or not os.listdir(BNVD_DIR):
    os.makedirs(BNVD_DIR, exist_ok=True)
    !kaggle datasets download -d sahabipin/bangladeshi-native-vehicle-dataset -p {BNVD_DIR} --unzip
else:
    print('BNVD already on Drive')
!du -sh {BNVD_DIR} && find {BNVD_DIR} -maxdepth 2 -type d | head -20

In [ ]:
# Poribohon-BD (CC BY 4.0) — Mendeley, no credentials needed.
PORIBOHON_DIR = f'{DATA_ROOT}/PoribohonBD'
import os
os.makedirs(PORIBOHON_DIR, exist_ok=True)
# Mendeley serves a versioned zip; resolve the current file URL from the DOI page
# https://data.mendeley.com/datasets/pwyyg8zmk5/2  (DOI 10.17632/pwyyg8zmk5.2)
print('Download the zip from the DOI page into:', PORIBOHON_DIR)

## 5 · Audit the real files (this is the evidence)

In [ ]:
from research.perception.dataset_audit import audit_yolo_dataset, audit_to_markdown
from research.perception.dataset_registry import BNVD_CLASSES, by_key
import json, os

bnvd = by_key('BNVD')
claimed = {k: bnvd.claim(k) for k in ('n_images','n_instances','n_classes','mean_bbox_per_image')}

# Point these at wherever the archive actually unpacked (printed in step 4).
audit = audit_yolo_dataset(
    root=BNVD_DIR,
    class_names=BNVD_CLASSES,      # order read from the AUTHORS' annotation_counter.py
    dataset_key='BNVD',
    claimed=claimed,
    hash_images=True,
)
os.makedirs(f'{ART_ROOT}/p1_perception', exist_ok=True)
audit.to_json(__import__('pathlib').Path(f'{ART_ROOT}/p1_perception/bnvd_audit.json'))
print(audit_to_markdown(audit))

In [ ]:
# Taxonomy coverage: what BNVD can and cannot train, and what stays undecided.
from dhakaflow.perception.taxonomy import coverage_report, open_decisions
import json
rep = coverage_report('BNVD')
print(json.dumps(rep, indent=2))
print('\nOPEN DECISIONS (resolve from real crops, never from the label name):')
for m in open_decisions('BNVD'):
    print(' -', m.source_label, '->', [c.value for c in m.candidate_targets])

In [ ]:
# Visual resolution of the two AMBIGUOUS traps: 'Van' and 'Easybike'.
# Crop real annotated instances and LOOK at them before deciding the mapping.
import random, pathlib
from PIL import Image
import matplotlib.pyplot as plt

def crop_class(root, class_index, class_name, n=8):
    root = pathlib.Path(root)
    labels = list(root.rglob('*.txt'))
    random.seed(0); random.shuffle(labels)
    crops = []
    for lp in labels:
        if len(crops) >= n: break
        for line in lp.read_text().splitlines():
            parts = line.split()
            if not parts or int(float(parts[0])) != class_index: continue
            ip = next((p for p in root.rglob(lp.stem + '.*')
                       if p.suffix.lower() in {'.jpg','.jpeg','.png'}), None)
            if not ip: continue
            try: im = Image.open(ip).convert('RGB')
            except Exception: continue
            cx, cy, w, h = (float(x) for x in parts[1:5])
            W, H = im.size
            box = (max(0,int((cx-w/2)*W)), max(0,int((cy-h/2)*H)),
                   min(W,int((cx+w/2)*W)), min(H,int((cy+h/2)*H)))
            if box[2]-box[0] > 10 and box[3]-box[1] > 10:
                crops.append(im.crop(box)); break
    if crops:
        fig, axes = plt.subplots(1, len(crops), figsize=(3*len(crops), 3))
        for ax, c in zip чи([axes] if len(crops)==1 else axes, crops):
            ax.imshow(c); ax.axis('off')
        fig.suptitle(f'{class_name} (index {class_index})'); plt.show()
    return crops

VAN_IDX, EASYBIKE_IDX = 15, 5   # from BNVD_CLASSES order
_ = crop_class(BNVD_DIR, VAN_IDX, 'Van')
_ = crop_class(BNVD_DIR, EASYBIKE_IDX, 'Easybike')

## 6 · Official RT-DETR at the pinned SHA

Official code, unmodified. The pin comes from `upstream/UPSTREAM_PINS.yaml`
(`RT-DETR @ 068dfde65f2667ad6555883c69d73de886518cad`, verified 2026-08-14).

In [ ]:
RTDETR_SHA = '068dfde65f2667ad6555883c69d73de886518cad'
import os, subprocess
if not os.path.exists('/content/RT-DETR'):
    !git clone -q https://github.com/lyuwenyu/RT-DETR.git /content/RT-DETR
!cd /content/RT-DETR && git checkout -q {RTDETR_SHA} && git rev-parse HEAD
# Confirm the pin actually matches before anything is called a reproduction.
actual = subprocess.run(['git','-C','/content/RT-DETR','rev-parse','HEAD'],
                        capture_output=True, text=True).stdout.strip()
assert actual == RTDETR_SHA, f'SHA mismatch: {actual}'
print('official RT-DETR pinned OK')
!ls /content/RT-DETR/rtdetrv2_pytorch/configs/rtdetrv2 2>/dev/null | head

## 7 · Next (do not run before the audit is reviewed)

Fine-tuning uses the **official** train script and **official** config with only
a dataset adapter changed — never a re-implemented model. The audit record
above must be reviewed first, because the class mapping decides what the
detector is even trained to predict.

## 8 · Detector candidates and the Bangladesh benchmark protocol

Fresh-qualified 2026-08-15 (all official repos verified to resolve):

| Candidate | Venue | Published COCO result | Repo |
|---|---|---|---|
| RT-DETR / RT-DETRv2 | CVPR 2024 | v2-S 48.1 AP @ 217 FPS | lyuwenyu/RT-DETR @ 068dfde6 |
| **D-FINE** | ICLR 2025 Spotlight | L: 54.0 AP @ 124 FPS (T4) | Peterande/D-FINE |
| **DEIM** (trains RT-DETR/D-FINE) | CVPR 2025 | DEIM-D-FINE-L: 54.7 AP @ 124 FPS, ~50% less training time | Intellindust-AI-Lab/DEIM |
| **RT-DETRv4** | ECCV 2026 | SOTA claims vs YOLOv10-13 | RT-DETRs/RT-DETRv4 |

**No winner is chosen until measured.** Every candidate runs the SAME protocol:
official code, official config, official pretrained checkpoint, fine-tuned on
the identical Bangladesh split, evaluated on the identical held-out set.

**Metrics (all mandatory):** rickshaw AP · CNG AP · easybike AP · motorcycle AP
· bus AP · overall mAP50 · mAP50:95 · precision · recall · FPS · latency (ms) ·
peak VRAM. Per-class AP on rickshaw/CNG/easybike is decisive — overall mAP can
hide failure on exactly the classes that make Dhaka different.

**DEIM note for our GPU budget:** DEIM's official repo re-implements RT-DETR and
D-FINE under one training framework with up to 50% faster convergence — if its
reproduction sanity passes, it becomes the shared training harness (same role
BasicTS plays for forecasting) and cuts the fine-tuning budget roughly in half.


In [ ]:
# Pin all four candidates at exact SHAs (official code only, never modified)
CANDIDATES = {
    'RT-DETR':   ('https://github.com/lyuwenyu/RT-DETR.git',        '068dfde65f2667ad6555883c69d73de886518cad'),
    'D-FINE':    ('https://github.com/Peterande/D-FINE.git',        None),  # resolve & record
    'DEIM':      ('https://github.com/Intellindust-AI-Lab/DEIM.git', None),
    'RT-DETRv4': ('https://github.com/RT-DETRs/RT-DETRv4.git',      None),
}
import subprocess, os, json
resolved = {}
for name, (url, sha) in CANDIDATES.items():
    dest = f'/content/{name}'
    if not os.path.exists(dest):
        subprocess.run(['git','clone','-q',url,dest], check=True)
    if sha:
        subprocess.run(['git','-C',dest,'checkout','-q',sha], check=True)
    head = subprocess.run(['git','-C',dest,'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip()
    resolved[name] = head
    print(f'{name:12} {head[:12]}')
# Persist the exact SHAs used — paste back into upstream/UPSTREAM_PINS.yaml
with open(f'{ART_ROOT}/p1_perception/candidate_shas.json','w') as f:
    json.dump(resolved, f, indent=2)

### License-clean path (independence from BNVD)

BNVD's license is UNRESOLVED (author permission requested in parallel). P1 is
therefore run on **two dataset tracks**:

1. **BNVD track** — strongest data, pending permission; results held until
   licensing is clear.
2. **License-clean track** — Poribohon-BD (**CC BY 4.0**, verified) converted
   VOC→YOLO via the audited mapping, so every P1 conclusion has a citable,
   redistribution-safe backing even if BNVD permission never arrives.

The winner must win on BOTH tracks or the discrepancy is itself reported.